In [3]:
"""
Professional Animated Model Comparison Visualizations (Fixed)
Save as: src/visualization/model_comparison_animated_fixed.py
Run: python src/visualization/model_comparison_animated_fixed.py
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Create directories
VIZ_PATH = Path("visualizations/model_comparison")
VIZ_PATH.mkdir(parents=True, exist_ok=True)
HTML_PATH = VIZ_PATH / 'html'
HTML_PATH.mkdir(parents=True, exist_ok=True)

# Professional color palette
COLORS = {
    'primary': '#1E88E5',
    'secondary': '#DC143C',
    'success': '#2ECC71',
    'warning': '#F39C12',
    'info': '#00ACC1',
    'dark': '#2C3E50',
    'light': '#ECF0F1',
    'purple': '#9B59B6',
    'orange': '#E67E22',
    'catboost': '#FF6B6B',
    'xgboost': '#4ECDC4',
    'lightgbm': '#45B7D1',
    'randomforest': '#96CEB4'
}


def plot_animated_radar_chart(metrics_df, save_path=None):
    """
    Create interactive radar chart comparing multiple models
    
    Parameters:
    - metrics_df: DataFrame with columns: model, rmse, mae, r2, direction_accuracy, sharpe_ratio
    - save_path: path to save the plot
    """
    print("\n🎯 Creating animated radar chart...")
    
    # Define metrics and their directions (1 = higher better, -1 = lower better)
    metrics = {
        'rmse': {'name': 'RMSE', 'direction': -1},
        'mae': {'name': 'MAE', 'direction': -1},
        'r2': {'name': 'R²', 'direction': 1},
        'direction_accuracy': {'name': 'Direction Accuracy', 'direction': 1},
        'sharpe_ratio': {'name': 'Sharpe Ratio', 'direction': 1}
    }
    
    # Normalize metrics (0-1 scale, higher is better)
    metrics_normalized = metrics_df.copy()
    
    for metric, config in metrics.items():
        values = metrics_df[metric].values
        if config['direction'] == -1:
            # Invert so lower is better becomes higher
            normalized = 1 - (values - values.min()) / (values.max() - values.min())
        else:
            normalized = (values - values.min()) / (values.max() - values.min())
        metrics_normalized[metric] = normalized
    
    fig = go.Figure()
    
    colors = [COLORS['catboost'], COLORS['xgboost'], COLORS['lightgbm'], COLORS['randomforest']]
    
    for idx, model in enumerate(metrics_df['model'].values):
        values = [metrics_normalized[metric].iloc[idx] for metric in metrics.keys()]
        values += values[:1]  # Close the loop
        
        categories = [config['name'] for config in metrics.values()]
        categories += categories[:1]
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories,
            fill='toself',
            name=model,
            line=dict(color=colors[idx % len(colors)], width=2),
            fillcolor=f'rgba({int(colors[idx % len(colors)][1:3], 16)}, ' +
                      f'{int(colors[idx % len(colors)][3:5], 16)}, ' +
                      f'{int(colors[idx % len(colors)][5:7], 16)}, 0.3)',
            hovertemplate='<b>%{theta}</b><br>Score: %{r:.3f}<extra></extra>'
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickfont=dict(size=10),
                tickformat='.0%'
            ),
            angularaxis=dict(
                tickfont=dict(size=12, weight='bold'),
                rotation=90
            )
        ),
        title=dict(
            text='<b>Model Performance Radar Chart</b><br><sub>Normalized metrics (higher is better)</sub>',
            x=0.5,
            font=dict(size=18, family='Arial Black')
        ),
        height=600,
        width=800,
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved radar chart to {save_path}")
    
    fig.write_html(HTML_PATH / 'radar_chart.html')
    
    return fig


def plot_animated_bar_chart(metrics_df, save_path=None):
    """
    Create interactive grouped bar chart comparing models
    
    Parameters:
    - metrics_df: DataFrame with model performance metrics
    - save_path: path to save the plot
    """
    print("\n📊 Creating animated bar chart...")
    
    metrics_config = {
        'rmse': {'title': 'RMSE (%)', 'color': COLORS['primary'], 'multiplier': 100},
        'mae': {'title': 'MAE (%)', 'color': COLORS['warning'], 'multiplier': 100},
        'direction_accuracy': {'title': 'Direction Accuracy', 'color': COLORS['success'], 'multiplier': 1},
        'sharpe_ratio': {'title': 'Sharpe Ratio', 'color': COLORS['purple'], 'multiplier': 1},
        'r2': {'title': 'R² Score', 'color': COLORS['info'], 'multiplier': 1}
    }
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[config['title'] for config in metrics_config.values()],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    models = metrics_df['model'].values
    colors_model = [COLORS['catboost'], COLORS['xgboost'], COLORS['lightgbm'], COLORS['randomforest']]
    
    for idx, (metric, config) in enumerate(metrics_config.items()):
        row = idx // 3 + 1
        col = idx % 3 + 1
        
        values = metrics_df[metric].values * config['multiplier']
        
        # Determine bar colors (highlight best performer)
        if metric in ['rmse', 'mae']:
            best_idx = np.argmin(values)
        else:
            best_idx = np.argmax(values)
        
        bar_colors = []
        for i, val in enumerate(values):
            if i == best_idx:
                bar_colors.append(COLORS['success'])
            else:
                bar_colors.append(config['color'])
        
        fig.add_trace(
            go.Bar(
                x=models,
                y=values,
                name=metric,
                marker_color=bar_colors,
                text=[f'{val:.3f}' for val in values],
                textposition='outside',
                hovertemplate='<b>%{x}</b><br>%{y:.3f}<extra></extra>',
                showlegend=False
            ),
            row=row, col=col
        )
        
        # Add horizontal line for average
        avg_val = np.mean(values)
        fig.add_hline(
            y=avg_val, line_dash="dash", line_color="gray",
            annotation_text=f'Avg: {avg_val:.3f}',
            annotation_position="bottom right",
            row=row, col=col
        )
    
    fig.update_layout(
        title=dict(
            text='<b>Model Performance Comparison</b><br><sub>Best performers highlighted in green</sub>',
            x=0.5,
            font=dict(size=18, family='Arial Black')
        ),
        height=700,
        width=1100,
        showlegend=False,
        template='plotly_white',
        bargap=0.2
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved bar chart to {save_path}")
    
    fig.write_html(HTML_PATH / 'bar_chart.html')
    
    return fig


def plot_interactive_performance_table(metrics_df, save_path=None):
    """
    Create interactive styled table with sorting and highlighting
    
    Parameters:
    - metrics_df: DataFrame with model performance metrics
    - save_path: path to save the plot
    """
    print("\n📋 Creating interactive performance table...")
    
    # Prepare display data
    display_df = metrics_df.copy()
    display_df['rmse'] = display_df['rmse'] * 100
    display_df['mae'] = display_df['mae'] * 100
    display_df['direction_accuracy'] = display_df['direction_accuracy'] * 100
    
    # Format values
    display_df['rmse'] = display_df['rmse'].map('{:.2f}%'.format)
    display_df['mae'] = display_df['mae'].map('{:.2f}%'.format)
    display_df['r2'] = display_df['r2'].map('{:.3f}'.format)
    display_df['direction_accuracy'] = display_df['direction_accuracy'].map('{:.1f}%'.format)
    display_df['sharpe_ratio'] = display_df['sharpe_ratio'].map('{:.2f}'.format)
    
    # Create table
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=['<b>Model</b>', '<b>RMSE</b>', '<b>MAE</b>', '<b>R²</b>', 
                   '<b>Direction Accuracy</b>', '<b>Sharpe Ratio</b>'],
            fill_color=COLORS['primary'],
            align='center',
            font=dict(color='white', size=12, family='Arial Black'),
            height=40
        ),
        cells=dict(
            values=[
                display_df['model'],
                display_df['rmse'],
                display_df['mae'],
                display_df['r2'],
                display_df['direction_accuracy'],
                display_df['sharpe_ratio']
            ],
            fill_color=[['#f8f9fa', '#e9ecef'] * len(display_df)],
            align='center',
            font=dict(color=COLORS['dark'], size=11),
            height=35,
            format=['', '.2%', '.2%', '.3f', '.1%', '.2f']
        )
    )])
    
    fig.update_layout(
        title=dict(
            text='<b>Model Performance Summary Table</b><br><sub>Interactive comparison of all models</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        height=400,
        width=1000,
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved performance table to {save_path}")
    
    fig.write_html(HTML_PATH / 'performance_table.html')
    
    return fig


def plot_parallel_coordinates(metrics_df, save_path=None):
    """
    Create parallel coordinates plot for multivariate analysis
    
    Parameters:
    - metrics_df: DataFrame with model performance metrics
    - save_path: path to save the plot
    """
    print("\n📈 Creating parallel coordinates plot...")
    
    # Normalize metrics for better visualization
    plot_df = metrics_df.copy()
    plot_df['rmse'] = plot_df['rmse'] * 100
    plot_df['mae'] = plot_df['mae'] * 100
    plot_df['direction_accuracy'] = plot_df['direction_accuracy'] * 100
    
    fig = px.parallel_coordinates(
        plot_df,
        dimensions=['rmse', 'mae', 'r2', 'direction_accuracy', 'sharpe_ratio'],
        color='rmse',
        color_continuous_scale=px.colors.diverging.RdYlGn_r,
        title='<b>Parallel Coordinates: Multi-dimensional Model Comparison</b>',
        labels={
            'rmse': 'RMSE (%)',
            'mae': 'MAE (%)',
            'r2': 'R² Score',
            'direction_accuracy': 'Direction Accuracy (%)',
            'sharpe_ratio': 'Sharpe Ratio'
        }
    )
    
    fig.update_layout(
        height=500,
        width=1000,
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved parallel coordinates to {save_path}")
    
    fig.write_html(HTML_PATH / 'parallel_coordinates.html')
    
    return fig


def plot_model_ranking_dashboard(metrics_df, save_path=None):
    """
    Create model ranking dashboard with score cards
    
    Parameters:
    - metrics_df: DataFrame with model performance metrics
    - save_path: path to save the plot
    """
    print("\n🏆 Creating model ranking dashboard...")
    
    # Calculate composite scores
    metrics_df = metrics_df.copy()
    metrics_df['rmse_score'] = 1 - (metrics_df['rmse'] - metrics_df['rmse'].min()) / (metrics_df['rmse'].max() - metrics_df['rmse'].min())
    metrics_df['mae_score'] = 1 - (metrics_df['mae'] - metrics_df['mae'].min()) / (metrics_df['mae'].max() - metrics_df['mae'].min())
    metrics_df['r2_score'] = (metrics_df['r2'] - metrics_df['r2'].min()) / (metrics_df['r2'].max() - metrics_df['r2'].min())
    metrics_df['dir_score'] = (metrics_df['direction_accuracy'] - metrics_df['direction_accuracy'].min()) / (metrics_df['direction_accuracy'].max() - metrics_df['direction_accuracy'].min())
    metrics_df['sharpe_score'] = (metrics_df['sharpe_ratio'] - metrics_df['sharpe_ratio'].min()) / (metrics_df['sharpe_ratio'].max() - metrics_df['sharpe_ratio'].min())
    
    metrics_df['composite_score'] = (metrics_df['rmse_score'] + metrics_df['mae_score'] + 
                                      metrics_df['r2_score'] + metrics_df['dir_score'] + 
                                      metrics_df['sharpe_score']) / 5
    
    # Sort by composite score
    metrics_df = metrics_df.sort_values('composite_score', ascending=False)
    
    # Create bar chart for composite scores
    fig = go.Figure()
    
    colors_model = [COLORS['catboost'], COLORS['xgboost'], COLORS['lightgbm'], COLORS['randomforest']]
    
    fig.add_trace(go.Bar(
        x=metrics_df['model'],
        y=metrics_df['composite_score'] * 100,
        marker_color=colors_model[:len(metrics_df)],
        text=metrics_df['composite_score'].apply(lambda x: f'{x:.1%}'),
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Composite Score: %{y:.1f}%<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text='<b>🏆 Model Ranking by Composite Score</b><br><sub>Weighted average of all performance metrics</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis=dict(title='Model'),
        yaxis=dict(title='Composite Score (%)', range=[0, 100]),
        height=500,
        width=800,
        template='plotly_white'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved ranking dashboard to {save_path}")
    
    fig.write_html(HTML_PATH / 'ranking_dashboard.html')
    
    # Print ranking
    print("\n🏆 Model Ranking:")
    for idx, row in metrics_df.iterrows():
        print(f"   {row['model']}: {row['composite_score']:.1%}")
    
    return fig


def plot_animated_bubble_chart(metrics_df, save_path=None):
    """
    Create animated bubble chart for model comparison
    
    Parameters:
    - metrics_df: DataFrame with model performance metrics
    - save_path: path to save the plot
    """
    print("\n🫧 Creating animated bubble chart...")
    
    # Prepare data
    plot_df = metrics_df.copy()
    plot_df['rmse'] = plot_df['rmse'] * 100
    plot_df['mae'] = plot_df['mae'] * 100
    plot_df['direction_accuracy'] = plot_df['direction_accuracy'] * 100
    
    fig = px.scatter(
        plot_df,
        x='rmse',
        y='direction_accuracy',
        size='sharpe_ratio',
        color='model',
        hover_name='model',
        size_max=50,
        text='model',
        title='<b>Model Performance Bubble Chart</b><br><sub>Size represents Sharpe Ratio</sub>',
        labels={
            'rmse': 'RMSE (%) (Lower Better)',
            'direction_accuracy': 'Direction Accuracy (%) (Higher Better)',
            'sharpe_ratio': 'Sharpe Ratio'
        },
        color_discrete_sequence=[COLORS['catboost'], COLORS['xgboost'], 
                                  COLORS['lightgbm'], COLORS['randomforest']]
    )
    
    fig.update_traces(
        textposition='top center',
        marker=dict(line=dict(width=2, color='white'))
    )
    
    fig.update_layout(
        height=500,
        width=900,
        template='plotly_white',
        hovermode='closest'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"✅ Saved bubble chart to {save_path}")
    
    fig.write_html(HTML_PATH / 'bubble_chart.html')
    
    return fig


def generate_all_model_comparisons(metrics_df, save_path=None):
    """
    Generate all model comparison visualizations
    
    Parameters:
    - metrics_df: DataFrame with model performance metrics
    - save_path: output directory (optional)
    """
    print("\n" + "="*60)
    print("🎨 Generating Complete Model Comparison Visualizations")
    print("="*60)
    
    # 1. Radar Chart
    plot_animated_radar_chart(metrics_df, save_path=HTML_PATH / 'radar_chart.html')
    
    # 2. Bar Chart
    plot_animated_bar_chart(metrics_df, save_path=HTML_PATH / 'bar_chart.html')
    
    # 3. Performance Table
    plot_interactive_performance_table(metrics_df, save_path=HTML_PATH / 'performance_table.html')
    
    # 4. Parallel Coordinates
    plot_parallel_coordinates(metrics_df, save_path=HTML_PATH / 'parallel_coordinates.html')
    
    # 5. Ranking Dashboard
    plot_model_ranking_dashboard(metrics_df, save_path=HTML_PATH / 'ranking_dashboard.html')
    
    # 6. Bubble Chart
    plot_animated_bubble_chart(metrics_df, save_path=HTML_PATH / 'bubble_chart.html')
    
    print("\n" + "="*60)
    print("✅ All model comparison visualizations generated successfully!")
    print(f"📁 Output directory: {HTML_PATH}")
    print("="*60)


if __name__ == "__main__":
    # Demo data
    demo_df = pd.DataFrame({
        'model': ['CatBoost', 'XGBoost', 'LightGBM', 'RandomForest'],
        'rmse': [0.0265, 0.0380, 0.0390, 0.0378],
        'mae': [0.0197, 0.0324, 0.0337, 0.0285],
        'r2': [0.942, 0.850, 0.830, 0.865],
        'direction_accuracy': [0.412, 0.373, 0.400, 0.434],
        'sharpe_ratio': [1.42, 1.22, 1.18, 1.35]
    })
    
    # Generate all visualizations
    generate_all_model_comparisons(demo_df)


🎨 Generating Complete Model Comparison Visualizations

🎯 Creating animated radar chart...
✅ Saved radar chart to visualizations\model_comparison\html\radar_chart.html

📊 Creating animated bar chart...
✅ Saved bar chart to visualizations\model_comparison\html\bar_chart.html

📋 Creating interactive performance table...
✅ Saved performance table to visualizations\model_comparison\html\performance_table.html

📈 Creating parallel coordinates plot...
✅ Saved parallel coordinates to visualizations\model_comparison\html\parallel_coordinates.html

🏆 Creating model ranking dashboard...
✅ Saved ranking dashboard to visualizations\model_comparison\html\ranking_dashboard.html

🏆 Model Ranking:
   CatBoost: 92.8%
   RandomForest: 49.8%
   XGBoost: 10.4%
   LightGBM: 8.9%

🫧 Creating animated bubble chart...
✅ Saved bubble chart to visualizations\model_comparison\html\bubble_chart.html

✅ All model comparison visualizations generated successfully!
📁 Output directory: visualizations\model_comparison\